# Лабораторная работа 1.3. Метод простых итераций и метод Зейделя

**Задание.** Реализовать метод простых итераций и метод Зейделя в виде программ, задавая в качестве
входных данных матрицу системы, вектор правых частей и точность вычислений. Используя разработанное ПО,
решить СЛАУ. Проанализировать количество итераций, необходимое для достижения заданной точности.

$$
\begin{cases}
18x_1 + 8x_2 - 3x_3 + 4x_4 = -84 \\
-7x_1 + 15x_2 - 5x_3 - 2x_4 = -5 \\
-4x_1 + 13x_3 + 4x_4 = -38 \\
-8x_1 - 8x_2 - 6x_3 + 31x_4 = 263
\end{cases}
$$

## Реализация

In [ ]:
import math


def norm_vec(v):
    return max(abs(x) for x in v)


def norm_mat(M):
    return max(sum(abs(x) for x in row) for row in M)


def to_iterative_form(A, b):
    """Приведение Ax = b к виду x = beta + alpha x."""
    n = len(A)
    for i in range(n):
        if A[i][i] == 0:
            raise ValueError(f"Нулевой диагональный элемент a[{i}][{i}] — переставьте уравнения")
    alpha = [[0.0 if i == j else -A[i][j] / A[i][i] for j in range(n)] for i in range(n)]
    beta = [b[i] / A[i][i] for i in range(n)]
    return alpha, beta


def simple_iterations(A, b, eps, max_iter=10_000):
    """Метод простых итераций. Возвращает (x, число итераций, история оценок погрешности)."""
    alpha, beta = to_iterative_form(A, b)
    n = len(beta)
    a_norm = norm_mat(alpha)
    coef = a_norm / (1 - a_norm) if a_norm < 1 else 1.0

    x = beta[:]
    history = []
    for k in range(1, max_iter + 1):
        x_new = [beta[i] + sum(alpha[i][j] * x[j] for j in range(n)) for i in range(n)]
        err = coef * norm_vec([x_new[i] - x[i] for i in range(n)])
        history.append(err)
        x = x_new
        if err <= eps:
            return x, k, history
    raise RuntimeError("Метод простых итераций не сошёлся за max_iter итераций")


def seidel(A, b, eps, max_iter=10_000):
    """Метод Зейделя. Возвращает (x, число итераций, история оценок погрешности)."""
    alpha, beta = to_iterative_form(A, b)
    n = len(beta)
    a_norm = norm_mat(alpha)
    c_norm = norm_mat([[alpha[i][j] if j >= i else 0.0 for j in range(n)] for i in range(n)])
    coef = c_norm / (1 - a_norm) if a_norm < 1 else 1.0

    x = beta[:]
    history = []
    for k in range(1, max_iter + 1):
        x_prev = x[:]
        for i in range(n):  # x[j] при j < i уже обновлены на этой итерации
            x[i] = beta[i] + sum(alpha[i][j] * x[j] for j in range(n))
        err = coef * norm_vec([x[i] - x_prev[i] for i in range(n)])
        history.append(err)
        if err <= eps:
            return x, k, history
    raise RuntimeError("Метод Зейделя не сошёлся за max_iter итераций")


def apriori_iterations(alpha, beta, eps):
    """Априорная оценка числа итераций метода простых итераций."""
    a, bt = norm_mat(alpha), norm_vec(beta)
    return math.ceil((math.log(eps) - math.log(bt) + math.log(1 - a)) / math.log(a) - 1)

## Исходные данные

In [18]:
A = [
    [18,  8, -3,  4],
    [-7, 15, -5, -2],
    [-4,  0, 13,  4],
    [-8, -8, -6, 31],
]
b = [-84, -5, -38, 263]
eps = 1e-6

for row, bi in zip(A, b):
    print("  [" + " ".join(f"{v:4}" for v in row) + f" ]   | {bi:5}")

  [  18    8   -3    4 ]   |   -84
  [  -7   15   -5   -2 ]   |    -5
  [  -4    0   13    4 ]   |   -38
  [  -8   -8   -6   31 ]   |   263


## Решение методом простых итераций

In [19]:
x_si, k_si, hist_si = simple_iterations(A, b, eps)
alpha, beta = to_iterative_form(A, b)
for i, xi in enumerate(x_si, 1):
    print(f"x{i} = {xi:.6f}")
print(f"\nИтераций: {k_si}   (априорная оценка: {apriori_iterations(alpha, beta, eps)})")

x1 = -5.000000
x2 = -4.000000
x3 = -6.000000
x4 = 5.000000

Итераций: 38   (априорная оценка: 270)


## Решение методом Зейделя

In [24]:
x_sd, k_sd, hist_sd = seidel(A, b, eps)
for i, xi in enumerate(x_sd, 1):
    print(f"x{i} = {xi:.6f}")
print(f"\nИтераций: {k_sd}")

x1 = -5.000000
x2 = -4.000000
x3 = -6.000000
x4 = 5.000000

Итераций: 21


## Анализ количества итераций

In [25]:
print(f"{'eps':>8} | {'апр. оценка':>11} | {'простые итер.':>13} | {'Зейдель':>7}")
print("-" * 50)
for p in range(1, 11):
    e = 10.0 ** -p
    _, k1, _ = simple_iterations(A, b, e)
    _, k2, _ = seidel(A, b, e)
    print(f"{e:8.0e} | {apriori_iterations(alpha, beta, e):11} | {k1:13} | {k2:7}")

     eps | апр. оценка | простые итер. | Зейдель
--------------------------------------------------
   1e-01 |         103 |            14 |       9
   1e-02 |         136 |            19 |      11
   1e-03 |         170 |            23 |      14
   1e-04 |         203 |            28 |      16
   1e-05 |         237 |            33 |      19
   1e-06 |         270 |            38 |      21
   1e-07 |         303 |            43 |      24
   1e-08 |         337 |            47 |      26
   1e-09 |         370 |            52 |      29
   1e-10 |         403 |            57 |      31
